# Assignment 1 – Regression Pipeline and Initial Model Development
**Dataset:** Sleep Efficiency Dataset  
**Target Variable:** `Sleep efficiency` (numerical, range 0–1)

## 1. Dataset Description

The Sleep Efficiency Dataset has 452 rows with information about people's sleep collected in 2021. Each row is one person, and the columns describe both their sleep physiology and lifestyle habits.

**Features (15 columns total):**

| Column | Type | Description |
|---|---|---|
| ID | Integer | row id, not useful |
| Age | Integer | age |
| Gender | Categorical | Male / Female |
| Bedtime | Datetime | when they went to bed |
| Wakeup time | Datetime | when they woke up |
| Sleep duration | Float | total sleep hours |
| **Sleep efficiency** | **Float** | **TARGET – ratio of actual sleep time to time in bed (0–1)** |
| REM sleep percentage | Integer | % of sleep in REM |
| Deep sleep percentage | Integer | % of sleep in deep stage |
| Light sleep percentage | Integer | % of sleep in light stage |
| Awakenings | Float | how many times they woke up during the night |
| Caffeine consumption | Float | mg of caffeine in 24h before sleep |
| Alcohol consumption | Float | alcohol units in 24h before sleep |
| Smoking status | Categorical | Yes / No |
| Exercise frequency | Float | days per week |

The dataset qualifies under the second criterion (200+ rows, 10+ features): 452 rows and 13 usable features after dropping ID.

## 2. Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('Sleep_Efficiency.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# check dtypes and missing values
print(df.dtypes)
print()
print(df.isnull().sum())

In [ ]:
# drop ID – it's just a row number, no predictive value
df.drop(columns=['ID'], inplace=True)

# bedtime and wakeup are datetime strings – need to extract something useful
# going with hour + minute as a float (e.g. 23:30 → 23.5)
# also using sin/cos encoding so 23:00 and 01:00 are treated as close
# (without this, hour 23 and hour 1 are 22 units apart which is wrong)
df['Bedtime'] = pd.to_datetime(df['Bedtime'])
df['Wakeup time'] = pd.to_datetime(df['Wakeup time'])

hour_b = df['Bedtime'].dt.hour + df['Bedtime'].dt.minute / 60
df['bedtime_sin'] = np.sin(2 * np.pi * hour_b / 24)
df['bedtime_cos'] = np.cos(2 * np.pi * hour_b / 24)

hour_w = df['Wakeup time'].dt.hour + df['Wakeup time'].dt.minute / 60
df['wakeup_sin'] = np.sin(2 * np.pi * hour_w / 24)
df['wakeup_cos'] = np.cos(2 * np.pi * hour_w / 24)

df.drop(columns=['Bedtime', 'Wakeup time'], inplace=True)
print(df.columns.tolist())

In [ ]:
# encode Gender and Smoking status (both binary so label encoding is fine)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])           # Female=0, Male=1
df['Smoking status'] = le.fit_transform(df['Smoking status'])  # No=0, Yes=1

print(df['Gender'].unique())
print(df['Smoking status'].unique())

In [ ]:
# handle missing values
# Awakenings, Caffeine, Alcohol, Exercise frequency all have some NaNs
# using median imputation – more robust than mean especially since
# Caffeine has an outlier at 200mg which would pull the mean up

print('missing before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

num_cols = df.select_dtypes(include='number').columns.tolist()
imp = SimpleImputer(strategy='median')
df[num_cols] = imp.fit_transform(df[num_cols])

print('missing after:', df.isnull().sum().sum())

In [ ]:
print('final shape:', df.shape)
df.describe().round(3)

**Summary of preprocessing steps:**
- Dropped `ID` – not a feature
- Extracted cyclic features from `Bedtime` and `Wakeup time` using sin/cos transformation. This handles the wrap-around at midnight (23:00 and 01:00 should be close, not 22 units apart)
- `Wakeup time` also kept as features instead of throwing it away
- Binary categorical variables encoded with LabelEncoder
- Median imputation for missing values (4 columns affected, 67 values total)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# viz 1 – distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Sleep efficiency'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Sleep Efficiency', fontsize=13)
axes[0].set_xlabel('Sleep Efficiency')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['Sleep efficiency'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot of Sleep Efficiency', fontsize=13)
axes[1].set_ylabel('Sleep Efficiency')
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig('viz1_target_distribution.png', bbox_inches='tight')
plt.show()

print(f'mean:   {df["Sleep efficiency"].mean():.3f}')
print(f'median: {df["Sleep efficiency"].median():.3f}')
print(f'std:    {df["Sleep efficiency"].std():.3f}')
print(f'min:    {df["Sleep efficiency"].min():.3f}')
print(f'max:    {df["Sleep efficiency"].max():.3f}')

The distribution is left-skewed – most people have relatively high sleep efficiency (peak around 0.88–0.92), but there's a clear tail on the left side going down to 0.50. The median (0.82) is higher than the mean which confirms the skew. The boxplot shows no extreme outliers, the lower values around 0.50 are real observations not errors.

In [ ]:
# viz 2 – correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('viz2_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('correlations with sleep efficiency:')
print(corr['Sleep efficiency'].drop('Sleep efficiency').sort_values(ascending=False).round(3))

**Key observations from the heatmap:**

- `Deep sleep percentage` has the strongest positive correlation with efficiency (~0.79). More deep sleep = better sleep.
- `Light sleep percentage` and `Awakenings` have the strongest negative correlations (-0.82 and -0.55).
- `Alcohol consumption` and `Smoking status` also negatively correlated.
- **Important issue:** REM + Deep + Light sleep percentages always sum to 100%, so they are perfectly multicollinear. The correlation of -0.99 between Deep and Light sleep confirms this. This will be a problem for linear regression (but not for Random Forest).

In [ ]:
# viz 3 – scatter plots for the most correlated features
key_features = ['Deep sleep percentage', 'Awakenings',
                'Alcohol consumption', 'Exercise frequency']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
colors = ['steelblue', 'tomato', 'goldenrod', 'mediumseagreen']

for ax, feat, color in zip(axes.flat, key_features, colors):
    ax.scatter(df[feat], df['Sleep efficiency'], alpha=0.4, s=20, color=color)
    m, b = np.polyfit(df[feat], df['Sleep efficiency'], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, m * x_line + b, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('Sleep Efficiency', fontsize=10)
    ax.set_title(f'{feat} vs Sleep Efficiency', fontsize=11)

plt.tight_layout()
plt.savefig('viz3_feature_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# viz 4 – group comparisons (smokers vs non-smokers, and awakenings groups)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

smoking_labels = {0: 'Non-smoker', 1: 'Smoker'}
for val, label in smoking_labels.items():
    subset = df[df['Smoking status'] == val]['Sleep efficiency']
    axes[0].hist(subset, bins=20, alpha=0.6, label=label)
axes[0].set_title('Sleep Efficiency by Smoking Status', fontsize=12)
axes[0].set_xlabel('Sleep Efficiency')
axes[0].legend()

df['Awakenings_group'] = pd.cut(df['Awakenings'], bins=[-1, 0, 1, 2, 10],
                                 labels=['0', '1', '2', '3+'])
df.boxplot(column='Sleep efficiency', by='Awakenings_group', ax=axes[1], patch_artist=True)
axes[1].set_title('Sleep Efficiency by Number of Awakenings')
axes[1].set_xlabel('Awakenings per Night')
axes[1].set_ylabel('Sleep Efficiency')
plt.suptitle('')

plt.tight_layout()
plt.savefig('viz4_group_comparison.png', bbox_inches='tight')
plt.show()

df.drop(columns=['Awakenings_group'], inplace=True)

**Patterns from the last two plots:**

1. Deep sleep has a clear positive relationship with efficiency – the strongest visual trend out of all 4 features.
2. Each additional awakening noticeably reduces efficiency. People with 0 awakenings cluster around 0.88+.
3. Alcohol has a moderate negative trend, though there's a lot of variance at each alcohol level.
4. Exercise shows a weak positive trend – helpful but not a strong predictor on its own.
5. Smokers look completely different from non-smokers – their distribution peaks at 0.50–0.55 while non-smokers peak at 0.90. This suggests a strong non-linear effect that correlation alone doesn't fully capture.

## 4. Data Splitting

In [ ]:
X = df.drop(columns=['Sleep efficiency'])
y = df['Sleep efficiency']

# 70/30 split, fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f'train: {X_train.shape[0]} samples')
print(f'test:  {X_test.shape[0]} samples')
print(f'features: {X_train.shape[1]}')

## 5. Model Training

In [ ]:
# Model 1 – Linear Regression
# need to fix the multicollinearity issue first:
# dropping Light sleep percentage since REM + Deep + Light = 100% always
# also scaling features so coefficients are on the same scale

X_train_lr = X_train.drop(columns=['Light sleep percentage'])
X_test_lr  = X_test.drop(columns=['Light sleep percentage'])

scaler = StandardScaler()
X_train_lr_s = scaler.fit_transform(X_train_lr)
X_test_lr_s  = scaler.transform(X_test_lr)

lr = LinearRegression()
lr.fit(X_train_lr_s, y_train)
y_pred_lr = lr.predict(X_test_lr_s)

coef_df = pd.DataFrame({'Feature': X_train_lr.columns, 'Coefficient': lr.coef_})
print(coef_df.sort_values('Coefficient', ascending=False).to_string(index=False))

In [ ]:
# Model 2 – Random Forest
# using max_depth=15 to avoid overfitting (default None grows trees fully)
# RF can use all features including Light sleep % since it handles correlated
# features naturally through random feature subsampling

rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

feat_imp = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_})
print(feat_imp.sort_values('Importance', ascending=False).to_string(index=False))

## 6. Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'\n{name}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  R²   : {r2:.4f}')
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2}

results = []
results.append(evaluate('Linear Regression', y_test, y_pred_lr))
results.append(evaluate('Random Forest',     y_test, y_pred_rf))

pd.DataFrame(results).set_index('Model').round(4)

In [ ]:
# cross-validation to get more reliable estimates (single split can be lucky/unlucky)
X_lr_full = scaler.fit_transform(X.drop(columns=['Light sleep percentage']))
cv_lr = cross_val_score(LinearRegression(), X_lr_full, y, cv=5, scoring='r2')
cv_rf = cross_val_score(
    RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
    X, y, cv=5, scoring='r2')

print(f'CV R²  LR: {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'CV R²  RF: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')

In [ ]:
# residual plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, name, color in zip(
        axes,
        [y_pred_lr, y_pred_rf],
        ['Linear Regression', 'Random Forest'],
        ['steelblue', 'tomato']):
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.4, s=20, color=color)
    ax.axhline(0, linestyle='--', color='black', linewidth=1)
    ax.set_xlabel('Predicted Sleep Efficiency')
    ax.set_ylabel('Residual')
    ax.set_title(f'{name} – Residuals', fontsize=12)

plt.tight_layout()
plt.savefig('viz5_residuals.png', bbox_inches='tight')
plt.show()

In [ ]:
# actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, name, color in zip(
        axes,
        [y_pred_lr, y_pred_rf],
        ['Linear Regression', 'Random Forest'],
        ['steelblue', 'tomato']):
    ax.scatter(y_test, y_pred, alpha=0.4, s=20, color=color)
    lims = [min(y_test.min(), y_pred.min()) - 0.02,
            max(y_test.max(), y_pred.max()) + 0.02]
    ax.plot(lims, lims, 'k--', linewidth=1.2)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Sleep Efficiency')
    ax.set_ylabel('Predicted Sleep Efficiency')
    ax.set_title(f'{name} – Actual vs Predicted', fontsize=12)

plt.tight_layout()
plt.savefig('viz6_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

In [ ]:
# feature importance for RF
feat_imp_sorted = feat_imp.sort_values('Importance')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp_sorted['Feature'], feat_imp_sorted['Importance'],
        color='tomato', edgecolor='white')
ax.set_title('Random Forest – Feature Importance', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('viz7_feature_importance.png', bbox_inches='tight')
plt.show()

## 7. Analysis – Model Comparison

| Metric | Linear Regression | Random Forest |
|--------|:-----------------:|:-------------:|
| RMSE   | 0.0614            | 0.0486        |
| MAE    | 0.0501            | 0.0368        |
| R²     | 0.8143            | 0.8840        |
| CV R² (5-fold) | 0.7975 ± 0.019 | 0.8644 ± 0.016 |

Random Forest outperforms Linear Regression on all three metrics. The difference holds up in cross-validation too, so it's not just a lucky test split.

**Why RF does better here:**

1. **Multicollinearity** – even after dropping Light sleep percentage, the remaining sleep stage features are still somewhat correlated. RF handles this naturally since it only uses a random subset of features at each split.

2. **Non-linear relationships** – looking at the scatter plots, some features don't follow a clean linear pattern. Alcohol for example seems to have a stronger effect at higher doses. Decision trees can pick up these thresholds without needing polynomial terms.

3. **Outliers** – Caffeine has a value of 200mg which is way above the rest. LR is sensitive to this, RF is not since splits are based on ordering not raw values.

The residual plots also show this clearly. LR has a visible upward bias at low predicted values (around 0.50–0.65) – it systematically over-predicts for people who actually sleep poorly. RF residuals are more randomly distributed around zero.

## 8. Conclusion

The goal was to build a regression pipeline to predict sleep efficiency from lifestyle and physiological data. Both models work reasonably well, but Random Forest (R² = 0.884) is clearly the better choice for this dataset.

**Main findings:**
- Deep sleep percentage and awakenings are by far the most important predictors (together ~85% of RF feature importance)
- Smoking has a strong non-linear effect that's easy to miss just from correlation (-0.33)
- Lifestyle factors like alcohol, caffeine and exercise matter but are secondary

**Things that could be improved:**
- Try XGBoost/LightGBM as a third model
- Hyperparameter tuning for Random Forest
- More data would help – 452 samples is enough but more would give better generalization